In [ ]:
### GPTDatasetV1

import torch
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        ## 1. 전체 텍스트 토큰화
        token_ids = tokenizer.encode(txt, allowed_special={"<!endoftext!>"})

        ## 2. input, target 토큰 생성
        for i in range(0, len(token_ids) - max_length, stride):
            inputs = token_ids[i : max_length]
            targets = token_ids[i + 1 : i + max_length + 1]

            self.input_ids.append(torch.tensor(inputs))
            self.target_ids.append(torch.tensor(targets))

    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [ ]:
### create_dataloader_v1()

import tiktoken

def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):
    ## 1. 토크나이저 생성
    tokenizer = tiktoken.get_encoding("gpt2)

    ## 2. 데이터 셋 생성
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    ## 3. 데이터 로더 생성
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last, num_workers=num_workers)

    return dataloader;


In [ ]:
### test dataloader

raw_text = "this is a raw text to make token"

dataloader = create_dataloader_v1(raw_text, batch_size=1, max_length=4, stride=1, shuffle=False, drop_last=False, num_workers=0)

data_iter = iter(dataloader)

batch1 = next(data_iter)
print( batch1 )

batch2 = next(data_iter)
print( batch2 )

In [ ]:
### GPT embedding
### input embedding = token_embedding + pos_embedding

vocab_size = 50257
out_dim = 256

max_length=4
content_length=max_length

## 1. embedding layer 생성
token_embedding_layer = torch.nn.Embedding( vocab_size, out_dim )
pos_embedding_layer = torch.nn.Embedding( content_length, out_dim )

## 2. dataloader 생성
raw_text = "this is a raw text to make token"
dataloader = create_dataloader_v1(raw_text, batch_size=2, max_length=max_length, stride=max_length, shuffle=False)

## 3. input embedding 생성
input, target = next(iter(dataloader))
token_embedding = token_embedding_layer(input)
pos_embedding = pos_embedding_layer(torch.arange(content_length))
input_embedding = token_embedding + pos_embedding

print(token_embedding.shape)    # [batch, content_length, out_dim]  = [2, 4, 256]
print(pos_embedding.shape)      # [content_length, out_dim]         = [4, 256]
print(input_embedding.shape)    # [batch, content_length, out_dim]  = [2, 4, 256]

In [ ]:
### tokenizer 로드하는 함수

from transformers import AutoTokenizer

# 1. qwen, gemma
AutoTokenizer.from_pretrained("Qwen/Qwen-3.0.6B")
AutoTokenzier.from_pretrained("google/gemma-3B")